In [1]:
import torch 
import numpy as np 
import pathlib
from pathlib import Path
import yaml
import os
import pickle
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint

import importlib
from src.location_classifier_lightning import LocationClassifier
import argparse
from argparse import ArgumentParser

In [6]:
config = 'config/binaural_attn/word_task_standard_v07.yaml'
ckpt_path = '/om2/user/imgriff/projects/torch_2_aud_attn/attn_cue_models/word_task_standard_v07/checkpoints/epoch=3-step=67111.ckpt'

In [7]:
seed_everything(123)
config_path = config

config_path = pathlib.Path(config_path)
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

with_projection, lim_train_batch = True, 5000
config['hparas']['lim_train_batch'] = lim_train_batch
# add transfer learning config
config['model']['num_classes']['num_locs'] = 504 # 360 azimuth and 90 in elevation
del config['model']['num_classes']['num_words']
config['corpus']['task'] = "location"
config['corpus']['skip_negative_elev'] = True
n_layers = 7
config['model']['n_layers'] = n_layers
config['model']['with_projection'] = with_projection
with_projection_str = 'with_projection' if with_projection else 'no_projection'
config['model']['projection_size'] = 256

config['hparas']['lr'] = 0.0005
config['num_workers'] = 1
checkpoint_path = ckpt_path
print(f"Training with config: {config_path.stem}")
print(f"Using checkpoint: {checkpoint_path}")
print(f"Using {n_layers} layers")

[rank: 0] Seed set to 123


Training with config: word_task_standard_v07
Using checkpoint: /om2/user/imgriff/projects/torch_2_aud_attn/attn_cue_models/word_task_standard_v07/checkpoints/epoch=3-step=67111.ckpt
Using 7 layers


In [8]:
config['hparas']['batch_size'] = 5

In [9]:
model = LocationClassifier(config, ckpt_path=checkpoint_path)
conv_modules = {name:module for name, module in model.model._orig_mod.model_dict.named_children() if 'conv' in name or 'relu' in name}
classifier_layer = list(conv_modules.keys())[-1]
print(f"Using classifier layer: {classifier_layer}")

Using BaseAuditoryAttentionForTransferV1
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram
Using classifier layer: conv_block_6


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


In [14]:
corpora_config = config['corpus']
hparas_config = config['hparas']

In [15]:
from corpus.binaural_attention_h5 import BinauralAttentionDataset

In [17]:
dataset = BinauralAttentionDataset(**corpora_config, batch_size=hparas_config['batch_size'], mode='train')

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v07
cue type: voice_and_location
985 files in train concat dataset


In [25]:
import src.audio_transforms as at

In [26]:
audio_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.BinauralCombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'], high_snr=config['noise_kwargs']['high_snr']),
                at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02), # 20 * np.log10(0.02/20e-6) = 60 dB SPL 
            ])

In [27]:
def collate_fn(samples):
        # samples is a single-element list holding a tuple batches
        samples = samples[0]
        aud_features, _ = audio_transforms(samples[0], None)
        labels = torch.from_numpy(samples[3]).type(torch.LongTensor)
        return aud_features, labels

In [28]:
dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=config['num_workers'], 
            collate_fn=collate_fn,
            pin_memory=True,
            # persistent_workers=True,
            shuffle=False,
        )

In [42]:
for data in dataloader:
    batch = data[0]
    labels = data[1]
    print(data[0].shape)
    break

torch.Size([5, 2, 110250])


In [38]:
# import audio display
import IPython.display as ipd

In [57]:
single_ex = batch[0]
ipd.Audio(single_ex[0].numpy(), rate=44100)

In [56]:
print(label_to_azim_elev(int(labels[0])))

(255, 0)


In [ ]:
def azim_elev_to_label(azim, elev):
    return np.array((((elev + 30) / 10) * 72) + (azim / 5), dtype=np.int64)

In [52]:
def label_to_azim_elev(label):
    elev = ((label // 72) * 10) - 30
    azim = (label % 72) * 5
    return azim, elev